## Data Preparation
- Import data
- Merge datasets into one dataframe
- Format data correctly as we want it.

In [1]:
import pandas as pd

In [2]:
df2019 = pd.read_csv("StationFootfall_2019.csv")
df2020 = pd.read_csv("StationFootfall_2020.csv")
df2021 = pd.read_csv("StationFootfall_2021.csv")
df2022 = pd.read_csv("StationFootfall_2022.csv")
df2023 = pd.read_csv("StationFootfall_2023.csv")
df2024 = pd.read_csv("StationFootfall_2024.csv")
df2025 = pd.read_csv("stationfootfall-2025.csv")

In [3]:
df2019 = df2019.rename(columns={"DayOFWeek": "DayOfWeek"})
df2020 = df2020.rename(columns={"DayOFWeek": "DayOfWeek"})
df2021 = df2021.rename(columns={"DayOFWeek": "DayOfWeek"})
df2022 = df2022.rename(columns={"DayOFWeek": "DayOfWeek"})

In [4]:
df = pd.concat([
    df2019,
    df2020,
    df2021,
    df2022,
    df2023,
    df2024,
    df2025
], ignore_index=True)

In [5]:
df = df.rename(columns = {'TravelDate': 'date'})
df = df.rename(columns = {'DayOfWeek': 'day'})
df = df.rename(columns = {'Station': 'station'})
df = df.rename(columns = {'EntryTapCount': 'enter_count'})
df = df.rename(columns = {'ExitTapCount': 'exit_count'})

In [6]:
df["total_count"] = df["enter_count"] + df["exit_count"]

In [7]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

In [8]:
df.duplicated( # Check for duplicates
    subset=["date", "station"]
).sum()

np.int64(0)

In [9]:
df[[ # descrptive statistics
    "enter_count",
    "exit_count",
    "total_count"
]].describe()

,enter_count,exit_count,total_count
count,1.160147e+06,1.160147e+06,1.160147e+06
mean,7.696729e+03,7.720585e+03,1.541731e+04
std,1.287758e+04,1.320902e+04,2.605442e+04
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.682000e+03,1.621000e+03,3.309000e+03
50%,3.718000e+03,3.620000e+03,7.346000e+03
75%,7.877000e+03,7.783000e+03,1.566500e+04
max,1.614980e+05,1.805380e+05,3.299810e+05


In [10]:
(df[["enter_count", "exit_count", "total_count"]] < 0).sum() # Check extreme values

enter_count    0
exit_count     0
total_count    0
dtype: int64

In [11]:
df["year"].unique() # Check years

array([2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], dtype=int32)

In [12]:
df["is_weekend"] = df["day"].isin(
    ["Saturday", "Sunday"]
)

In [13]:
df["month_name"] = pd.to_datetime( # Add month names
    df["month"], format="%m"
).dt.month_name()

In [14]:
df["covid_period"] = df["year"].apply( # COVID variable
    lambda x: "Pre-COVID" if x == 2019
    else "COVID" if x in [2020, 2021]
    else "Post-COVID"
)

In [15]:
df.groupby("station")["total_count"].sum() # Total by station for the past 5+ years

station
Abbey Road DLR              3739565
Abbey Wood                 48393456
Acton Central              10382962
Acton Main Line            10518774
Acton Town                 34126759
                             ...   
Woodside Park              14799229
Woolwich Arsenal DLR       48159637
Woolwich Arsenal NR        10405467
Woolwich EL                34382192
Woolwich Elizabeth Line    15184090
Name: total_count, Length: 439, dtype: int64

In [ ]:
df.groupby("station")["total_count"].mean() # Daily Average by station

station
Abbey Road DLR              1377.372007
Abbey Wood                 17759.066422
Acton Central               3820.074319
Acton Main Line             3958.891231
Acton Town                 12551.216992
                               ...     
Woodside Park               5420.963004
Woolwich Arsenal DLR       17777.643780
Woolwich Arsenal NR         3845.331486
Woolwich EL                30891.457323
Woolwich Elizabeth Line    38150.979899
Name: total_count, Length: 439, dtype: float64

In [ ]:
annual = df.groupby( # Yearly average by station
    ["station", "year"]
)["total_count"].mean().reset_index()